In [ ]:
%pip install segmentation-models-pytorch timm opencv-python pandas
%pip uninstall -y torch torchvision
%pip install --no-cache-dir torch==2.7.0 torchvision==0.22.0 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
import os, cv2, torch, random
import numpy as np, pandas as pd
import albumentations as A
import segmentation_models_pytorch as smp
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
# from torch.optim.swa_utils import AveragedModel, SWALR
from tqdm.auto import tqdm
from scipy.optimize import minimize_scalar

In [ ]:
# ================= НАСТРОЙКИ (ПОД KAGGLE) =================
IMG_SIZE = 352
BATCH_SIZE = 24
NUM_EPOCHS = 30
LR = 5e-4
WEIGHT_DECAY = 5e-4
THRESHOLD = 0.5
DEVICE = 'cuda'

DATA_DIR = Path('/kaggle/input/competitions/dl-lab-3-product-segmentation/train')
IMAGES_DIR = DATA_DIR / 'images'
MASKS_DIR = DATA_DIR / 'masks'
SPLIT_PATH = Path('/kaggle/input/datasets/aaaaa982/dl-lab3-best/split.csv')
SAVE_DIR = Path('/kaggle/working')
PREV_CHECKPOINT_PATH = '/kaggle/input/datasets/aaaaa982/dl-lab3-best/best.pth'

# ================= ИНИЦИАЛИЗАЦИЯ =================
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision('high')

seed = 42
random.seed(seed); np.random.seed(seed)
torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

In [ ]:
# ================= ДАТАСЕТ И АУГМЕНТАЦИИ =================
class FastSegDataset(Dataset):
    def __init__(self, df, img_dir, mask_dir, tfms=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.mask_dir = Path(mask_dir)
        self.tfms = tfms
        self.preproc_fn = smp.encoders.get_preprocessing_fn("tu-convnext_base", "imagenet")

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        stem = Path(self.df.iloc[idx]['image']).stem
        img_path = next(self.img_dir.glob(f"{stem}.*"))
        mask_path = self.mask_dir / f"{stem}.png"

        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        if self.tfms:
            aug = self.tfms(image=img, mask=mask)
            img, mask = aug["image"], aug["mask"]

        img = self.preproc_fn(img.astype(np.float32)).transpose(2, 0, 1)
        mask = (mask > 0).astype(np.float32)[None, ...]
        return torch.from_numpy(img).float(), torch.from_numpy(mask).float()

train_tfm = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5), 
    A.VerticalFlip(p=0.5), 
    A.RandomRotate90(p=0.5),
    A.Affine(translate_percent=(-0.05, 0.05), scale=(0.9, 1.1), rotate=(-15, 15), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    # A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(1, 32), hole_width_range=(1, 32), p=0.3),
])
val_tfm = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE)])

df = pd.read_csv(SPLIT_PATH)
sub_col = 'subset' if 'subset' in df.columns else 'split'
train_df = df[df[sub_col].str.lower() == 'train']
val_df = df[df[sub_col].str.lower().isin(['val', 'validation', 'test'])]

train_loader = DataLoader(FastSegDataset(train_df, IMAGES_DIR, MASKS_DIR, train_tfm), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(FastSegDataset(val_df, IMAGES_DIR, MASKS_DIR, val_tfm), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# ================= МОДЕЛЬ И ФАЙНТЮН =================
model = smp.DeepLabV3Plus(
    encoder_name="tu-convnext_base", 
    encoder_weights=None,
    classes=1
)

head_conv = model.segmentation_head[0]
model.segmentation_head[0] = torch.nn.Sequential(
    head_conv,
    torch.nn.Dropout2d(0.05)
)

model = model.to(DEVICE)

if os.path.exists(PREV_CHECKPOINT_PATH):
    print(f"Загружаем чекпоинт: {PREV_CHECKPOINT_PATH}")
    ckpt = torch.load(PREV_CHECKPOINT_PATH, map_location=DEVICE)
    state_dict = ckpt.get('model_state_dict', ckpt)

    msg = model.load_state_dict(state_dict, strict=True)
    print(f"Успешно загружено: {msg}")

# ================= ЛОССЫ, ОПТИМИЗАТОР, SWA =================
dice_fn = smp.losses.DiceLoss('binary', from_logits=True)
focal_fn = smp.losses.FocalLoss('binary')
lovasz_fn = smp.losses.LovaszLoss('binary', from_logits=True)

def calc_loss(pred, target): 
    return 0.4*lovasz_fn(pred, target) + 0.4*dice_fn(pred, target) + 0.2*focal_fn(pred, target)

def get_dice(logits, targets):
    p = (torch.sigmoid(logits) > THRESHOLD).float().flatten(1)
    t = targets.flatten(1)
    return ((2.0 * (p * t).sum(1)) / (p.sum(1) + t.sum(1) + 1e-7)).mean().item()

opt = torch.optim.AdamW([
    {'params': model.encoder.parameters(), 'lr': LR * 0.1}, 
    {'params': model.decoder.parameters(), 'lr': LR},
    {'params': model.segmentation_head.parameters(), 'lr': LR},
], weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NUM_EPOCHS)
scaler = torch.amp.GradScaler('cuda')

# swa_model = AveragedModel(model)
# swa_start = 21
# swa_sched = SWALR(opt, swa_lr=LR * 0.1)

In [ ]:
# ================= ЦИКЛ ОБУЧЕНИЯ =================
best_dice = 0
history = []

for ep in range(1, NUM_EPOCHS + 1):
    model.train()
    tr_loss, tr_dice = 0, 0
    
    for x, y in tqdm(train_loader, desc=f"Ep {ep}/{NUM_EPOCHS}", leave=False):
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        
        with torch.amp.autocast('cuda'):
            pred = model(x)
            loss = calc_loss(pred, y)
            
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        
        tr_loss += loss.item(); tr_dice += get_dice(pred.detach(), y)

    # # SWA логика
    # if ep >= swa_start:
    #     swa_model.update_parameters(model)
    #     swa_sched.step()
    #     val_model = swa_model
    # else:
    #     sched.step()
    #     val_model = model
    
    sched.step()
    val_model = model

    # Валидация
    val_model.eval()
    vl_loss, vl_dice = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.amp.autocast('cuda'):
                pred = val_model(x)
            vl_loss += calc_loss(pred, y).item(); vl_dice += get_dice(pred, y)

    tr_loss, tr_dice = tr_loss / len(train_loader), tr_dice / len(train_loader)
    vl_loss, vl_dice = vl_loss / len(val_loader), vl_dice / len(val_loader)
    current_lr = opt.param_groups[0]['lr']

    print(f"Ep {ep:02d} | LR: {current_lr:.6f} | Train Loss: {tr_loss:.4f} Dice: {tr_dice:.4f} | Val Loss: {vl_loss:.4f} Dice: {vl_dice:.4f}")

    if vl_dice > best_dice:
        best_dice = vl_dice
        torch.save({'model_state_dict': val_model.state_dict()}, SAVE_DIR / "best_model.pth")
        print("Новый бест сохранен!")

    history.append({'epoch': ep, 'lr': current_lr, 'train_loss': tr_loss, 'train_dice': tr_dice, 'val_loss': vl_loss, 'val_dice': vl_dice})

pd.DataFrame(history).to_csv(SAVE_DIR / "history.csv", index=False)
print("Обучение завершено. Чекпоинты лежат в /kaggle/working/")

In [ ]:
# model = smp.DeepLabV3Plus(
#     encoder_name="tu-convnext_base", 
#     encoder_weights=None,
#     classes=1
# )

# head_conv = model.segmentation_head[0]
# model.segmentation_head[0] = torch.nn.Sequential(
#     head_conv,
#     torch.nn.Dropout2d(0.05)
# )

# model = model.to(DEVICE)

# p = Path('/kaggle/working/best_model.pth')
# if os.path.exists(p):
#     print(f"Загружаем чекпоинт: {p}")
#     ckpt = torch.load(p, map_location=DEVICE)
#     state_dict = ckpt.get('model_state_dict', ckpt)
#     msg = model.load_state_dict(state_dict, strict=True)
#     print(f"Успешно загружено: {msg}")


In [ ]:
@torch.no_grad()
def find_optimal_threshold(model, loader, device):
    model.eval()
    all_probs, all_masks = [], []
    
    for x, y in tqdm(loader, desc="Сбор предиктов с TTA"):
        x = x.to(device)
        # 4 TTA (Orig, HFlip, VFlip, Rot180)
        p1 = torch.sigmoid(model(x))
        p2 = torch.flip(torch.sigmoid(model(torch.flip(x, [3]))), [3])
        p3 = torch.flip(torch.sigmoid(model(torch.flip(x, [2]))), [2])
        p4 = torch.rot90(torch.sigmoid(model(torch.rot90(x, 2, [2, 3]))), k=-2, dims=[2, 3])
        
        tta_prob = (p1 + p2 + p3 + p4) / 4.0
        all_probs.append(tta_prob.cpu().numpy())
        all_masks.append(y.numpy())

    probs = np.concatenate(all_probs, axis=0)
    masks = np.concatenate(all_masks, axis=0)

    def calc_dice(t):
        preds = (probs > t).astype(np.float32)
        inter = (preds * masks).sum(axis=(1, 2, 3))
        denom = preds.sum(axis=(1, 2, 3)) + masks.sum(axis=(1, 2, 3))
        return ((2.0 * inter + 1e-7) / (denom + 1e-7)).mean()

    print("Подбор тресхолда...")
    res = minimize_scalar(lambda t: -calc_dice(t), bounds=(0.25, 0.7), method='bounded', options={'xatol': 0.005})
    
    print(f"Оптимальный порог: {res.x:.3f} | Dice: {-res.fun:.4f}")
    return res.x

val_loader_safe = DataLoader(val_loader.dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
best_threshold = find_optimal_threshold(model, val_loader_safe, DEVICE)

In [ ]:
# ================= САБМИШН =================

import os, cv2, json, torch
import numpy as np, pandas as pd
import segmentation_models_pytorch as smp
from pathlib import Path
from tqdm.auto import tqdm

# ================= НАСТРОЙКИ =================
TEST_IMAGES_DIR = Path('/kaggle/input/competitions/dl-lab-3-product-segmentation/test_images')
CHECKPOINT_PATH = "/kaggle/working/best_model.pth"
OUTPUT_CSV = "submission.csv"

IMG_SIZE = 352
THRESHOLD = 0.444  # HARDCODE
DEVICE = "cuda"

# ================= ИНИЦИАЛИЗАЦИЯ =================
print("Сборка модели и загрузка весов...")
model = smp.DeepLabV3Plus(
    encoder_name="tu-convnext_base",
    encoder_weights=None,
    classes=1
)
model.segmentation_head[0] = torch.nn.Sequential(
    model.segmentation_head[0],
    torch.nn.Dropout2d(0.05)
)

ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt.get('model_state_dict', ckpt), strict=True)
model.to(DEVICE)
model.eval()

preproc_fn = smp.encoders.get_preprocessing_fn("tu-convnext_base", "imagenet")

def serialize_mask(mask2d: np.ndarray) -> str:
    return json.dumps(mask2d.tolist(), separators=(",", ":"))

# ================= ИНФЕРЕНС С TTA =================
image_paths = sorted([p for p in TEST_IMAGES_DIR.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}])
rows = []

print(f"Старт инференса. Картинок: {len(image_paths)}, TTA: 6")

with torch.no_grad():
    for img_path in tqdm(image_paths, desc="Submission"):
        img = cv2.imread(str(img_path))
        if img is None: continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]

        inp = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR).astype(np.float32)
        inp = preproc_fn(inp)
        inp = torch.from_numpy(inp.transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)

        with torch.amp.autocast('cuda'):
            p1 = torch.sigmoid(model(inp))[0, 0].cpu().numpy()
            p2 = np.flip(torch.sigmoid(model(torch.flip(inp, [3])))[0, 0].cpu().numpy(), axis=1)
            p3 = np.flip(torch.sigmoid(model(torch.flip(inp, [2])))[0, 0].cpu().numpy(), axis=0)
            p4 = np.rot90(torch.sigmoid(model(torch.rot90(inp, 1, [2, 3])))[0, 0].cpu().numpy(), k=-1)
            p5 = np.rot90(torch.sigmoid(model(torch.rot90(inp, 3, [2, 3])))[0, 0].cpu().numpy(), k=-3)
    
            inp_diag = inp.permute(0, 1, 3, 2)
            p6 = torch.sigmoid(model(inp_diag))[0, 0].cpu().numpy().T

        pred = (p1 + p2 + p3 + p4 + p5 + p6) / 6.0

        if pred.shape != (H, W):
            pred = cv2.resize(pred, (W, H), interpolation=cv2.INTER_LINEAR)

        mask = (pred > THRESHOLD).astype(np.uint8)
        rows.append({'ImageId': img_path.name, 'mask': serialize_mask(mask)})

pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
print(f"Готово! Файл сохранен: /kaggle/working/{OUTPUT_CSV}")

In [ ]:
import os, cv2, torch
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
from pathlib import Path
from tqdm.auto import tqdm

# ================= НАСТРОЙКИ =================
NUM_WORST = 32 # колво худших
IMG_SIZE = 352
THRESHOLD = 0.444
DEVICE = 'cuda'

DATA_DIR = Path('/kaggle/input/competitions/dl-lab-3-product-segmentation/train')
IMAGES_DIR = DATA_DIR / 'images'
MASKS_DIR = DATA_DIR / 'masks'
SPLIT_PATH = Path('/kaggle/input/datasets/aaaaa982/dl-lab3-best/split.csv')
CHECKPOINT_PATH = "/kaggle/working/best_model.pth"

# ================= ЗАГРУЗКА МОДЕЛИ =================
model = smp.DeepLabV3Plus(encoder_name="tu-convnext_base", encoder_weights=None, classes=1)
model.segmentation_head[0] = torch.nn.Sequential(model.segmentation_head[0], torch.nn.Dropout2d(0.05))
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE).get('model_state_dict', {}), strict=False)
model.to(DEVICE).eval()
preproc_fn = smp.encoders.get_preprocessing_fn("tu-convnext_base", "imagenet")

# ================= ФУНКЦИЯ ОЦЕНКИ =================
def get_single_dice(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    union = pred.sum() + gt.sum()
    return 1.0 if union == 0 and inter == 0 else (2.0 * inter + 1e-7) / (union + 1e-7)

# ================= ПОИСК ХУДШИХ =================
df = pd.read_csv(SPLIT_PATH)
sub_col = 'subset' if 'subset' in df.columns else 'split'
val_df = df[df[sub_col].str.lower().isin(['val', 'validation', 'test'])].reset_index(drop=True)

results = []

print("Анализ валидационного сета...")
with torch.no_grad():
    for idx, row in tqdm(val_df.iterrows(), total=len(val_df)):
        stem = Path(row['image']).stem
        img_path = next(IMAGES_DIR.glob(f"{stem}.*"))
        mask_path = MASKS_DIR / f"{stem}.png"
        
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        gt_binary = (gt_mask > 0).astype(np.uint8)
    
        inp = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR).astype(np.float32)
        inp = torch.from_numpy(preproc_fn(inp).transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)
        
        with torch.amp.autocast('cuda'):
            pred_logits = model(inp)[0, 0].cpu().numpy()
            
        pred_mask = cv2.resize(pred_logits, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_LINEAR)
        pred_binary = (pred_mask > 0).astype(np.uint8) # Тут юзаем 0 (логит), что эквивалентно 0.5 вероятности
        
        dice = get_single_dice(pred_binary, gt_binary)
        results.append((dice, img_path, mask_path, pred_mask))

results.sort(key=lambda x: x[0])

In [ ]:
# ================= ВИЗУАЛИЗАЦИЯ =================
print(f"\nОтрисовка ТОП-{NUM_WORST} худших предсказаний...")
fig, axes = plt.subplots(NUM_WORST, 3, figsize=(15, 4 * NUM_WORST))

for i in range(NUM_WORST):
    dice, img_p, msk_p, pred_logits = results[i]
    
    img = cv2.cvtColor(cv2.imread(str(img_p)), cv2.COLOR_BGR2RGB)
    gt = cv2.imread(str(msk_p), cv2.IMREAD_GRAYSCALE)
    pred_bin = (pred_logits > 0).astype(np.uint8)
    
    # Оригинал
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f"Image: {img_p.stem}\nDice: {dice:.4f}")
    axes[i, 0].axis('off')
    
    # Ground Truth (Реальная)
    axes[i, 1].imshow(gt, cmap='gray')
    axes[i, 1].set_title("Ground Truth")
    axes[i, 1].axis('off')
    
    # Предсказание
    axes[i, 2].imshow(pred_bin, cmap='gray')
    axes[i, 2].set_title(f"Prediction (Thresh=0)")
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig('worst_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================= ВАРИАНТ ПОСТПРОЦЕССИНГА 1 =================

def postprocess_mask(mask_binary, min_area=300, keep_largest_n=2):
    if mask_binary.sum() == 0: return mask_binary
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask_clean = cv2.morphologyEx(mask_binary, cv2.MORPH_CLOSE, kernel)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask_clean, connectivity=8)
    if num_labels <= 1: return mask_clean
    areas = [(i, stats[i, cv2.CC_STAT_AREA]) for i in range(1, num_labels)]
    valid_areas = [x for x in areas if x[1] >= min_area]
    valid_areas.sort(key=lambda x: x[1], reverse=True)
    top_areas = valid_areas[:keep_largest_n]
    valid_labels = set([x[0] for x in top_areas])
    final_mask = np.zeros_like(mask_clean)
    for lbl in valid_labels:
        final_mask[labels == lbl] = 1
    return final_mask

# ================= ТЕСТ ПОСТПРОЦЕССИНГА 1 НА ВАЛЕ =================
print("Считаем Dice ДО и ПОСЛЕ чистки...")

raw_dices = []
clean_dices = []

for _, img_p, msk_p, pred_logits in tqdm(results, desc="Post-processing Test"):
    gt = cv2.imread(str(msk_p), cv2.IMREAD_GRAYSCALE)
    gt_bin = (gt > 0).astype(np.uint8)

    pred_bin = (pred_logits > 0).astype(np.uint8)
    raw_dice = get_single_dice(pred_bin, gt_bin)
    raw_dices.append(raw_dice)

    clean_bin = postprocess_mask(pred_bin, min_area=400, keep_largest_n=2)
    clean_dice = get_single_dice(clean_bin, gt_bin)
    clean_dices.append(clean_dice)

old_mean = np.mean(raw_dices)
new_mean = np.mean(clean_dices)

print(f"\n--- РЕЗУЛЬТАТЫ ---")
print(f"Сырой Validation Dice: {old_mean:.5f}")
print(f"Чистый Validation Dice: {new_mean:.5f}")
diff = new_mean - old_mean
if diff > 0:
    print(f"Морфа дала буст: +{diff:.5f}")
else:
    print(f"Стало хуже: {diff:.5f}")

In [ ]:
# ================= ВАРИАНТ ПОСТПРОЦЕССИНГА 2 =================

import cv2
import numpy as np
from tqdm.auto import tqdm

raw_dices = []
blur3_dices = []
blur5_dices = []

for _, _, msk_p, pred_logits in tqdm(results, desc="Median Blur Test"):
    gt = cv2.imread(str(msk_p), cv2.IMREAD_GRAYSCALE)
    gt_bin = (gt > 0).astype(np.uint8)
    pred_bin = (pred_logits > 0).astype(np.uint8)
    raw_dices.append(get_single_dice(pred_bin, gt_bin))
    mb3 = cv2.medianBlur(pred_bin, 3)
    blur3_dices.append(get_single_dice(mb3, gt_bin))
    mb5 = cv2.medianBlur(pred_bin, 5)
    blur5_dices.append(get_single_dice(mb5, gt_bin))

mean_raw = np.mean(raw_dices)
mean_b3 = np.mean(blur3_dices)
mean_b5 = np.mean(blur5_dices)

print(f"\n--- РЕЗУЛЬТАТЫ MEDIAN BLUR ---")
print(f"Сырой Dice:       {mean_raw:.5f}")
print(f"Блюр 3x3 Dice:    {mean_b3:.5f} (Разница: {mean_b3 - mean_raw:+.5f})")
print(f"Блюр 5x5 Dice:    {mean_b5:.5f} (Разница: {mean_b5 - mean_raw:+.5f})")

best_diff = max(mean_b3 - mean_raw, mean_b5 - mean_raw)
if best_diff > 0:
    print(f"МБлюр дал буст")
else:
    print(f"Опять мимо. Вопросы к разметчики...")

In [ ]:
import os, cv2, json, torch
import numpy as np, pandas as pd
import segmentation_models_pytorch as smp
from pathlib import Path
from tqdm.auto import tqdm

# ================= НАСТРОЙКИ =================
TEST_IMAGES_DIR = Path("/kaggle/input/competitions/dl-lab-3-product-segmentation/test_images")
CHECKPOINT_PATH = "/kaggle/working/best_model.pth"
OUTPUT_CSV = "mb_submission.csv"

IMG_SIZE = 352
THRESHOLD = 0.444 
DEVICE = "cuda"

# ================= МОДЕЛЬ =================
model = smp.DeepLabV3Plus(encoder_name="tu-convnext_base", encoder_weights=None, classes=1)
model.segmentation_head[0] = torch.nn.Sequential(
    model.segmentation_head[0], 
    torch.nn.Dropout2d(0.05)
)
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt.get('model_state_dict', ckpt))
model.to(DEVICE).eval()

preproc_fn = smp.encoders.get_preprocessing_fn("tu-convnext_base", "imagenet")

def serialize_mask(mask2d: np.ndarray) -> str:
    return json.dumps(mask2d.tolist(), separators=(",", ":"))

# ================= ИНФЕРЕНС =================
image_paths = sorted([p for p in TEST_IMAGES_DIR.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}])
rows = []

print(f"Запуск инференса: {len(image_paths)} фото | TTA: 6 | MedianBlur: 3x3")

with torch.no_grad():
    for img_path in tqdm(image_paths):
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None: continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        H, W = img_rgb.shape[:2]

        inp = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR).astype(np.float32)
        inp = torch.from_numpy(preproc_fn(inp).transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)

        with torch.amp.autocast('cuda'):
            p1 = torch.sigmoid(model(inp))[0, 0].cpu().numpy()
            p2 = np.flip(torch.sigmoid(model(torch.flip(inp, [3])))[0, 0].cpu().numpy(), axis=1)
            p3 = np.flip(torch.sigmoid(model(torch.flip(inp, [2])))[0, 0].cpu().numpy(), axis=0)
            p4 = np.rot90(torch.sigmoid(model(torch.rot90(inp, 1, [2, 3])))[0, 0].cpu().numpy(), k=-1)
            p5 = np.rot90(torch.sigmoid(model(torch.rot90(inp, 3, [2, 3])))[0, 0].cpu().numpy(), k=-3)
            p6 = torch.sigmoid(model(inp.permute(0, 1, 3, 2)))[0, 0].cpu().numpy().T
        pred = (p1 + p2 + p3 + p4 + p5 + p6) / 6.0
        if pred.shape != (H, W):
            pred = cv2.resize(pred, (W, H), interpolation=cv2.INTER_LINEAR)

        mask = (pred > THRESHOLD).astype(np.uint8)
        mask = cv2.medianBlur(mask, 3) 

        rows.append({'ImageId': img_path.name, 'mask': serialize_mask(mask)})

pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
print(f"Готово! Файл {OUTPUT_CSV} создан.")